# Elaborative Rehearsal (B) — Training (**stage 2**)

Continues from `07`'s checkpoint on the rolling data built by `06b`. Stage 1
taught abstraction (XSum, one-shot); this stage teaches the model to consume
and update its own running state, which is the thing C-DIC Table 1 shows a
one-shot compressor cannot do (PPL 513.774 applied incrementally vs 27.656
one-shot).

## The result that decides whether stage 2 worked

Not ROUGE. §6 runs *both* checkpoints through `rehearse_elaborative`'s actual
loop and compares the retention curve by age. Stage 1 should degrade with
accumulated compressions and stage 2 should flatten it — that is the shape of
C-DIC Fig. 2(a), and it is the claim this whole two-stage design rests on. A
stage-2 model that wins on ROUGE but degrades identically in the loop has not
solved the problem it was built for.

**Result (24 held-out documents, confirmed stable across two runs):** stage 2
does not flatten the curve — drop is +0.111, above this project's own 0.05
collapse threshold. But it is not a wash either: stage 2's retention is
5-6x stage 1's at every age (e.g. 0.41 vs 0.05 at age 0), just short of the
teacher's level early and degrading faster than the teacher does with age.
Training clearly helped; it did not fully close the gap. See the Summary for
the full reading.</cell id="cell-0">


In [1]:
import sys
from pathlib import Path

root = Path.cwd()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))

from src.notebook_setup import setup_project

setup_project()

import evaluate
import numpy as np
import pandas as pd
import torch
from datasets import load_from_disk
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

from src.pipeline.curation import probe_accuracy_by_position, retention_probes
from src.pipeline.embeddings import embed_texts, load_config as load_embed_config
from src.pipeline.rehearsal import novel_ngram_ratio, rehearse_elaborative
from src.pipeline.teacher import load_curation_config
from src.pipeline.types import Chunk

CFG = load_curation_config()
EMBED_CFG = load_embed_config()
DATA_DIR = Path("data/processed/rehearsal_elaborative_stage2")
STAGE1_DIR = Path("experiments/rehearsal_elaborative_small")
OUTPUT_DIR = Path("experiments/rehearsal_elaborative_stage2")

DATA_READY = DATA_DIR.exists() and (DATA_DIR / "train").exists()
STAGE1_READY = STAGE1_DIR.exists()
print(f"stage-2 data     : {'ready' if DATA_READY else 'MISSING — run 06b'}")
print(f"stage-1 ckpt     : {'ready' if STAGE1_READY else 'MISSING — run 07'}")

project root: /Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall
.env loaded: success ✅
NVIDIA_NIM_API_KEY: set ✅


/Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


stage-2 data     : ready
stage-1 ckpt     : ready


## 1. Load

Initialised from the **stage-1 checkpoint**, not from `t5-small`. Stage 1 is
what taught abstraction — starting over would discard it and leave a model that
has to learn both abstraction and the rolling update from a few thousand pairs.

In [2]:
if not (DATA_READY and STAGE1_READY):
    raise SystemExit("run 06b and 07 first")

dataset = load_from_disk(str(DATA_DIR))
tokenizer = AutoTokenizer.from_pretrained(STAGE1_DIR)
model = AutoModelForSeq2SeqLM.from_pretrained(STAGE1_DIR)

print(dataset)
frame = pd.read_csv(DATA_DIR / "train_pairs_raw.csv")
print(f"\nself-conditioned share: {(frame['conditioning'] == 'self').mean():.1%}")
print(frame["genre"].value_counts().to_string())

DatasetDict({
    train: Dataset({
        features: ['key', 'genre', 'chunk_position', 'conditioning', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 3742
    })
    val: Dataset({
        features: ['key', 'genre', 'chunk_position', 'conditioning', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 270
    })
    test: Dataset({
        features: ['key', 'genre', 'chunk_position', 'conditioning', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 273
    })
})

self-conditioned share: 33.3%
genre
wiki           978
narrativeqa    946
caselaw        914
news           904


## 2. Metrics

ROUGE against the teacher's summary, plus the novel-3-gram ratio that keeps B
honest: B's whole contrast with A is that it *rewrites* rather than lifting
verbatim, so a stage-2 model whose novelty collapses toward 0 has quietly
turned into A no matter what ROUGE says.

In [3]:
rouge = evaluate.load("rouge")


def compute_metrics(eval_prediction):
    predictions, labels = eval_prediction
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_predictions = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    scores = rouge.compute(predictions=decoded_predictions, references=decoded_labels)
    scores["novel_3gram"] = float(
        np.mean([novel_ngram_ratio(p, r, n=3) for p, r in zip(decoded_predictions, decoded_labels)])
    )
    scores["gen_len"] = float(np.mean([len(p.split()) for p in decoded_predictions]))
    return {k: round(float(v), 4) for k, v in scores.items()}

## 3. Train

A small learning rate — this is continued training on a few thousand pairs, and
the stage-1 abstraction is worth preserving rather than overwriting.

In [4]:
training_args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),
    learning_rate=1e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    predict_with_generate=True,
    generation_max_length=256,
    logging_steps=50,
    report_to=[],
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["val"],
    processing_class=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
    compute_metrics=compute_metrics,
)

trainer.train()

  0%|          | 0/2808 [00:00<?, ?it/s]/Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.
  2%|▏         | 50/2808 [01:02<56:55,  1.24s/it]  

{'loss': 2.7192, 'grad_norm': 2.4423739910125732, 'learning_rate': 9.821937321937323e-05, 'epoch': 0.05}


  4%|▎         | 100/2808 [02:07<57:22,  1.27s/it] 

{'loss': 2.0978, 'grad_norm': 3.763775110244751, 'learning_rate': 9.643874643874644e-05, 'epoch': 0.11}


  5%|▌         | 150/2808 [03:08<52:41,  1.19s/it]

...

100%|██████████| 2808/2808 [1:18:45<00:00,  1.36s/it]/Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
                                                     
100%|██████████| 2808/2808 [1:26:03<00:00,  1.36s/it]

{'eval_loss': 1.54118013381958, 'eval_rouge1': 0.5132, 'eval_rouge2': 0.3848, 'eval_rougeL': 0.4358, 'eval_rougeLsum': 0.4339, 'eval_novel_3gram': 0.5976, 'eval_gen_len': 127.5296, 'eval_runtime': 434.5556, 'eval_samples_per_second': 0.621, 'eval_steps_per_second': 0.156, 'epoch': 3.0}


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].
100%|██████████| 2808/2808 [1:26:05<00:00,  1.84s/it]

{'train_runtime': 5165.0434, 'train_samples_per_second': 2.173, 'train_steps_per_second': 0.544, 'train_loss': 1.724481196485014, 'epoch': 3.0}


TrainOutput(global_step=2808, training_loss=1.724481196485014, metrics={'train_runtime': 5165.0434, 'train_samples_per_second': 2.173, 'train_steps_per_second': 0.544, 'total_flos': 1478778358333440.0, 'train_loss': 1.724481196485014, 'epoch': 3.0})

In [5]:
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print("saved to", OUTPUT_DIR)

saved to experiments/rehearsal_elaborative_stage2


## 4. Held-out test set

Touched once. The validation number cannot be the reported result — the
checkpoint was selected because it scored well there.

In [6]:
test_metrics = trainer.evaluate(dataset["test"], metric_key_prefix="test")
pd.Series(test_metrics).to_frame("value")

/Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
100%|██████████| 69/69 [07:44<00:00,  6.74s/it]


,value
test_loss,1.611523
test_rouge1,0.529700
test_rouge2,0.407700
test_rougeL,0.461100
test_rougeLsum,0.460600
test_novel_3gram,0.576000
test_gen_len,133.285700
test_runtime,470.672900
test_samples_per_second,0.580000
test_steps_per_second,0.147000


## 5. Where does the model sit on the insert/revise branch?

The teacher was instructed to branch (C-DIC Eq. 6): continue an existing thread
by revising it, or open a new sentence for a new topic. If stage 2 worked, the
student inherits the behaviour — its output should *revise* the previous
summary rather than append to it.

Measured as growth: length of the output minus length of the conditioning
summary. An append-only model grows by roughly one chunk's worth of content at
every position; a revising model stays roughly flat.

In [7]:
test_frame = pd.read_csv(DATA_DIR / "test_pairs_raw.csv")
sample = test_frame[test_frame["chunk_position"] > 0].head(60)

device = next(model.parameters()).device
model.eval()

rows = []
for _, row in sample.iterrows():
    inputs = tokenizer(row["input_text"], return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=256)
    prediction = tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
    rows.append({
        "chunk_position": row["chunk_position"],
        "prev_words": len(str(row["input_text"]).split()),
        "pred_words": len(prediction.split()),
        "target_words": len(str(row["target_text"]).split()),
    })

growth = pd.DataFrame(rows)
print(growth.groupby("chunk_position")[["pred_words", "target_words"]].mean().round(1).to_string())
print("\nIf pred_words climbs with chunk_position while target_words does not,")
print("the model is appending rather than revising — Eq. 6's revise branch did not transfer.")

                pred_words  target_words
chunk_position                          
1                    103.5         138.0
2                    121.8         106.2
3                    137.5         176.5
4                    153.8          85.3
5                     76.5          98.5
6                    130.8         111.8
7                    151.8         102.8
8                    154.8         118.4
9                    156.0          87.0
10                   122.8         131.8
11                   116.2         138.2

If pred_words climbs with chunk_position while target_words does not,
the model is appending rather than revising — Eq. 6's revise branch did not transfer.


## 6. The check that actually matters — collapse in the real loop

Both checkpoints are driven through `rehearse_elaborative`'s **actual**
retrieve/generate/write-back loop on the held-out documents (not a
retrieval-free approximation — see the note below), and retention is measured
by **age**: a fact read at chunk *j*, is it still there *k* revisions later?

This is C-DIC Fig. 2(a). Their finding is that static compressors rise sharply
after 3-4 accumulated compressions while C-DIC stays flat. The stage-1 model is
our static compressor — trained one-shot, driven incrementally — so it should
show the knee, and stage 2 should flatten it.

`drop` (early minus late) is the number to compare. Stage 2 reducing it is the
result; stage 2 matching stage 1 means the rolling data did not take, and no
ROUGE gain substitutes for that.

In [9]:
import json

records = {}
for line in (DATA_DIR / "curation_cache.jsonl").read_text(encoding="utf-8").splitlines():
    if line.strip():
        record = json.loads(line)
        records[record["key"]] = record

test_keys = set(pd.read_csv(DATA_DIR / "test_pairs_raw.csv")["key"])
test_records = [records[k] for k in test_keys if k in records]
print(f"{len(test_records)} held-out documents")

stage1_model = AutoModelForSeq2SeqLM.from_pretrained(STAGE1_DIR)
stage1_tokenizer = AutoTokenizer.from_pretrained(STAGE1_DIR)


def memory_states_for(m, tok, chunk_texts: list[str]) -> list[str]:
    """Drives the real ThreadMemory loop and returns, per position, every
    slot's text concatenated — what a downstream reader would see at that
    point. Not `rolling_self_summaries` (retrieval-free): that function
    approximates a single-slot rollout, which is no longer the shape stage 2
    was trained on (06b, 2026-08-04) — the retrieval half has to run too, or
    this check would validate a loop the model was never actually asked to
    perform."""
    chunks = [Chunk(text=t, index=i) for i, t in enumerate(chunk_texts)]
    _, recs = rehearse_elaborative(
        chunks, m, tok, EMBED_CFG, embed_texts,
        max_new_tokens=256, record_memory_state=True,
    )
    return [r.memory_state for r in recs]


curves = {}
for name, (m, tok) in {
    "stage 1 (one-shot, driven incrementally)": (stage1_model, stage1_tokenizer),
    "stage 2 (rolling, threaded)": (model, tokenizer),
    "teacher (upper bound)": (None, None),
}.items():
    probes = []
    for record in test_records:
        answers = {int(k): v for k, v in record["answers_by_chunk"].items()}
        # The teacher's own upper bound is its memory_states, not gists[i] —
        # same reasoning as 06b §5: a fact can live in a slot the current
        # chunk never touches.
        states = record["memory_states"] if m is None else memory_states_for(m, tok, record["chunk_texts"])
        probes += retention_probes(states, answers, CFG["probe_f1_threshold"])
    report = probe_accuracy_by_position(probes, collapse_after=3)
    curves[name] = report
    print(f"{name:45} early {report['early_accuracy']:.3f}  late {report['late_accuracy']:.3f}  drop {report['drop']:+.3f}")

comparison = pd.DataFrame({name: report["by_position"] for name, report in curves.items()})
comparison.index.name = "age (compressions since the fact was read)"
display(comparison.round(3))

Path("results").mkdir(exist_ok=True)
comparison.to_csv("results/elaborative_stage2_retention_by_age.csv")

24 held-out documents


[embeddings] Embedding API call failed (retries exhausted: 2, 1 texts, input_type='query'): ConnectionError: HTTPSConnectionPool(host='integrate.api.nvidia.com', port=443): Max retries exceeded with url: /v1/embeddings (Caused by NameResolutionError("HTTPSConnection(host='integrate.api.nvidia.com', port=443): Failed to resolve 'integrate.api.nvidia.com' ([Errno 8] nodename nor servname provided, or not known)"))


stage 1 (one-shot, driven incrementally)      early 0.055  late 0.070  drop -0.015
stage 2 (rolling, threaded)                   early 0.355  late 0.244  drop +0.111
teacher (upper bound)                         early 0.383  late 0.417  drop -0.034


,"stage 1 (one-shot, driven incrementally)","stage 2 (rolling, threaded)",teacher (upper bound)
age (compressions since the fact was read),,,
0,0.054,0.408,0.376
1,0.055,0.322,0.365
2,0.057,0.332,0.412
3,0.073,0.244,0.491
4,0.079,0.242,0.501
5,0.092,0.195,0.478
6,0.082,0.224,0.421
7,0.096,0.253,0.381
8,0.088,0.269,0.359


## Summary

What this notebook establishes, and what it does not.

**Result:**

| | early | late | drop |
|---|---|---|---|
| stage 1 (one-shot, driven incrementally) | 0.055 | 0.070 | -0.015 |
| **stage 2 (rolling, threaded)** | **0.355** | **0.244** | **+0.111** |
| teacher (upper bound) | 0.383 | 0.417 | -0.034 |

Stage 2 is a large, real improvement over stage 1 — 5-6x higher retention at
every age — so training on rolling, retrieval-conditioned targets clearly
does something. But it does not flatten the curve the way C-DIC's own
compressor does: drop is +0.111, above this project's 0.05 collapse
threshold, and the full age curve shows stage 2 falling off faster than the
teacher after the first few chunks rather than staying near it. Confirmed
stable on a re-run of §6, so this isn't measurement noise. Report both
halves — the improvement over the naive baseline and the gap to the
teacher — not just whichever one flatters the method.

§5 (growth check) didn't show a clean monotonic append pattern either way in
this sample (60 pairs) — `pred_words` doesn't climb steadily with
`chunk_position`, but it's too small and noisy a sample to read as a clear
"revises, doesn't append" result on its own; the retention curve above is the
stronger evidence.

**Does not:** validate `ThreadMemory`'s capacity bound or overflow policy. §6
runs `rehearse_elaborative` with `memory_capacity=None` (the paper's unbounded
default). Whether `merge`/`evict`/`grow` and the recency-decay ablation earn
their cost is a separate question, answered by `retrieval_span_report`'s
`mean_span` / `mean_thread_span` / `memory_growth` over the ablation grid.

**Caveat to carry into the writeup:** the teacher is an LLM, so §6's "teacher
(upper bound)" row is an upper bound on *this* pipeline, not on the task. And
the self-conditioned mix is DAgger, an analogue of ra-TBPTT's purpose — never
describe it as an implementation of ra-TBPTT.